# Experiment 7: Dataset Generalization

Tests whether the meta-encoder framework generalizes across datasets:
CIFAR-10 (baseline), CIFAR-100, and STL-10.

Evaluates all 5 success criteria for each dataset.

**Data on Drive** (`My Drive/ctls/data/raw/`):
- `cifar-10-batches-py/`  — CIFAR-10 (already extracted)
- `cifar-100-python/`     — CIFAR-100 (already extracted)

## Colab Setup
Clones the repo, installs dependencies, and mounts Google Drive.

In [ ]:
import os, sys

# 1. Clone repository
REPO_DIR = '/content/trainable-circuits'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/jacobposchl/trainable-circuits {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# 2. Install extra dependencies
!pip install hdbscan umap-learn -q

# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 4. Path constants
DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data/raw'   # root for torchvision datasets
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print('Repo:     ', REPO_DIR)
print('Data:     ', DATA_DIR)
print('Checkpts: ', CKPT_DIR)

In [ ]:
import copy
import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt
from models.backbone import FrozenBackbone
from models.meta_encoder import MetaEncoder, ProfileRegressor
from data.cifar import get_standard_loaders
from evaluation.circuit_analysis import CircuitAnalyzer
from evaluation.discovery import SpanCentricDiscovery
from evaluation.metrics import (
    profile_reconstruction_r2,
    geometric_consistency,
    within_span_elevation,
    circuit_diversity,
    class_purity_distribution,
)

In [ ]:
def evaluate_dataset(config, checkpoint_path, device, data_dir, max_samples=2000, max_pairs=50_000):
    """Load a trained model and evaluate all 5 criteria. Returns a results dict."""
    config = copy.deepcopy(config)
    config['data']['data_dir'] = data_dir

    backbone = FrozenBackbone(
        arch=config['model']['arch'],
        num_classes=config['model']['num_classes'],
        pretrained=config['model']['pretrained'],
    ).to(device)

    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        projection_dim=config['model']['meta_encoder']['projection_dim'],
        n_heads=config['model']['meta_encoder']['n_heads'],
        n_transformer_layers=config['model']['meta_encoder']['n_transformer_layers'],
        dropout=config['model']['meta_encoder']['dropout']
    ).to(device)

    regressor = ProfileRegressor(
        input_dim=config['model']['meta_encoder']['projection_dim'],
        hidden_dim=config['model']['regressor']['hidden_dim']
    ).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
    regressor.load_state_dict(ckpt['regressor_state'])
    meta_encoder.eval()
    regressor.eval()

    _, val_loader = get_standard_loaders(
        data_dir=data_dir,
        batch_size=config['data'].get('batch_size', 256),
        num_workers=0,
        augment=False,
        download=False,
    )

    analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
    data     = analyzer.collect_representations(max_samples=max_samples)

    # Pair indices
    N = data['z_list'][0].shape[0]
    idx_a, idx_b = torch.triu_indices(N, N, offset=1)
    if idx_a.shape[0] > max_pairs:
        perm  = torch.randperm(idx_a.shape[0])[:max_pairs]
        idx_a, idx_b = idx_a[perm], idx_b[perm]

    true_sims    = CircuitAnalyzer.compute_pair_profiles(data['trajectories'], idx_a, idx_b)
    true_sims_np = true_sims.numpy()
    pair_indices = np.stack([idx_a.numpy(), idx_b.numpy()], axis=1)
    L            = len(data['z_list'])

    # z-space sims
    z_sims_np = np.zeros((idx_a.shape[0], L))
    for l in range(L):
        z_sims_np[:, l] = (
            data['z_list'][l][idx_a].numpy() * data['z_list'][l][idx_b].numpy()
        ).sum(axis=1)

    # Criterion 1
    preds = []
    with torch.no_grad():
        for l in range(L):
            z_prod = data['z_list'][l][idx_a] * data['z_list'][l][idx_b]
            preds.append(regressor(z_prod.to(device)).cpu())
    c1 = profile_reconstruction_r2(torch.stack(preds, dim=1).numpy(), true_sims_np)

    # Criterion 2
    c2 = geometric_consistency(z_sims_np, true_sims_np, L)

    # Criteria 3-5 via discovery
    disc_cfg  = config.get('discovery', {})
    discovery = SpanCentricDiscovery(
        n_layers=L,
        tau_discovery=disc_cfg.get('tau_discovery', 0.5),
        min_cluster_fraction=disc_cfg.get('min_cluster_fraction', 0.01),
        min_cluster_size=disc_cfg.get('hdbscan_min_cluster_size', 5),
    )
    circuits   = discovery.discover_all(true_sims_np, pair_indices)
    labels_np  = data['labels'].numpy()

    # Criterion 3: within-span elevation
    all_pass_c3 = True
    for circuit in circuits:
        l0, l1 = circuit['span']
        pop = true_sims_np[:, l0:l1+1].mean(axis=1)
        clu = pop[circuit['pair_mask']]
        if not within_span_elevation(clu, pop)['passes']:
            all_pass_c3 = False
    c3 = {'passes': all_pass_c3, 'n_circuits': len(circuits)}

    # Criterion 4
    c4 = circuit_diversity([c['span'] for c in circuits], L)

    # Criterion 5
    purities = [SpanCentricDiscovery.compute_class_purity(c, pair_indices, labels_np)
                for c in circuits]
    c5 = class_purity_distribution(purities) if purities else {'passes': False, 'n_agnostic': 0, 'n_specific': 0}

    return {'c1': c1, 'c2': c2, 'c3': c3, 'c4': c4, 'c5': c5}

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Dataset configurations
# torchvision looks for cifar-10-batches-py / cifar-100-python under root=DATA_DIR
# TODO: Add configs and checkpoints for CIFAR-100 and STL-10 once trained
dataset_experiments = {
    'cifar10': {
        'config':     CONFIG_DIR + '/phase1.yaml',
        'checkpoint': CKPT_DIR  + '/phase1/best.pt',
        'num_classes': 10,
    },
    'cifar100': {
        'config':     None,   # TODO
        'checkpoint': None,
        'num_classes': 100,
    },
    'stl10': {
        'config':     None,   # TODO
        'checkpoint': None,
        'num_classes': 10,
    },
}

In [ ]:
# Run evaluation for each dataset
all_results = {}

for dataset, exp in dataset_experiments.items():
    if exp['config'] is None or exp['checkpoint'] is None:
        print(f'Skipping {dataset} (not yet trained)')
        continue
    if not os.path.exists(exp['checkpoint']):
        print(f'Skipping {dataset}: checkpoint not found')
        continue

    print(f'\n=== {dataset.upper()} ===')
    with open(exp['config']) as f:
        config = yaml.safe_load(f)
    config['model']['num_classes'] = exp['num_classes']

    res = evaluate_dataset(config, exp['checkpoint'], device, DATA_DIR)
    all_results[dataset] = res

    print(f"  C1 R\u00b2:      {res['c1']['r2']:.4f}  {'PASS' if res['c1']['passes'] else 'FAIL'}")
    print(f"  C2 mean \u03c1: {res['c2']['mean_rho']:.4f}  {'PASS' if res['c2']['passes'] else 'FAIL'}")
    print(f"  C3 elev:    {res['c3']['n_circuits']} circuits  {'PASS' if res['c3']['passes'] else 'FAIL'}")
    print(f"  C4 cov:     {res['c4']['coverage']:.1%}  {'PASS' if res['c4']['passes'] else 'FAIL'}")
    print(f"  C5 bimodal: agnostic={res['c5']['n_agnostic']}, specific={res['c5']['n_specific']}  "
          f"{'PASS' if res['c5']['passes'] else 'FAIL'}")

In [ ]:
# Summary bar chart (C1 R² and C2 mean ρ)
if all_results:
    datasets = list(all_results.keys())
    r2s      = [all_results[d]['c1']['r2']       for d in datasets]
    rhos     = [all_results[d]['c2']['mean_rho'] for d in datasets]

    x     = np.arange(len(datasets))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width/2, r2s,  width, label='R\u00b2',          color='steelblue')
    ax.bar(x + width/2, rhos, width, label='Mean Spearman \u03c1', color='coral')
    ax.axhline(0.7,  color='steelblue', linestyle='--', alpha=0.6, label='R\u00b2 target')
    ax.axhline(0.65, color='coral',     linestyle='--', alpha=0.6, label='\u03c1 target')
    ax.set_xticks(x)
    ax.set_xticklabels(datasets)
    ax.set_title('Dataset Generalization')
    ax.legend()
    plt.tight_layout()
    plt.show()